In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from sklearn.datasets import make_blobs
from src.models import KMeansMember
from src.ensemble import AdaptiveClusteringEnsemble
from sklearn.metrics import adjusted_rand_score

In [3]:
# Generate a synthetic dataset (n=15 instances, 4 features)
X_synth, true_labels = make_blobs(n_samples=15, centers=3, n_features=4, random_state=42)

# Initialize the separated architecture
kmeans_base = KMeansMember()

ace_framework = AdaptiveClusteringEnsemble(
    base_member=kmeans_base,
    m_partitions=4,
    k_clusters=3,
    alpha_1=0.8,
    alpha_2=0.7,
    delta_alpha=0.1,
    random_state=123
)

# Run Stage 1
print("Running Stage 1: Matrix Transformation...")
theta_1_initial = ace_framework.stage1_transformation(X_synth)
print(f"Generated {theta_1_initial.shape[1]} initial base clusters.")

# Run Stage 2
print("Running Stage 2: Adaptive Consensus Merging...")
theta_1_final, theta_2_elim = ace_framework.stage2_consensus_merging(theta_1_initial, X_synth.shape[0])
print(f"Active consensus clusters remaining: {theta_1_final.shape[1]}")

# Run Stage 3
print("Running Stage 3: Hard Clustering Enforcement...")
final_labels = ace_framework.stage3_hard_clustering(theta_1_final, theta_2_elim)

# Validation against Ground Truth
ari_score = adjusted_rand_score(true_labels, final_labels)
print("\n======================================")
print(f"Final ACE Clustering Complete!")
print(f"Adjusted Rand Index (ARI): {ari_score:.4f}")
print("======================================")

Running Stage 1: Matrix Transformation...
Generated 12 initial base clusters.
Running Stage 2: Adaptive Consensus Merging...
Active consensus clusters remaining: 3
Running Stage 3: Hard Clustering Enforcement...

Final ACE Clustering Complete!
Adjusted Rand Index (ARI): 1.0000


In [11]:
from sklearn.datasets import load_iris
from sklearn.metrics import normalized_mutual_info_score
import numpy as np
from src.data_pipeline import DataPreprocessor

print("Initializing ACE Baseline Test on Real Data...\n")

# Load Real Dataset (Iris)
print("Loading Iris dataset...")
iris = load_iris()
X_raw, true_labels = iris.data, iris.target

# Preprocessing: Z-score standardization
preprocessor = DataPreprocessor()
X_scaled = preprocessor.process_dataset(X_raw)
k_true = len(np.unique(true_labels))

# Run K-Means Baseline
print("\n--- Running Standard ACE (K-Means) ---")
ace_kmeans = AdaptiveClusteringEnsemble(
    base_member=KMeansMember(),
    m_partitions=10, k_clusters=k_true, random_state=123
)
labels_kmeans = ace_kmeans.fit_predict(X_scaled)
print(f"K-Means ARI: {adjusted_rand_score(true_labels, labels_kmeans):.4f}")
print(f"K-Means NMI: {normalized_mutual_info_score(true_labels, labels_kmeans):.4f}")

Initializing ACE Baseline Test on Real Data...

Loading Iris dataset...

--- Running Standard ACE (K-Means) ---
K-Means ARI: 0.5829
K-Means NMI: 0.6352


In [12]:
from src.models import FCMMember
print("\n--- Running Proposed Fuzzy-ACE (FCM) ---")
ace_fcm = AdaptiveClusteringEnsemble(
    base_member=FCMMember(m_fuzz=2.0),
    m_partitions=10, k_clusters=k_true, random_state=123
)
labels_fcm = ace_fcm.fit_predict(X_scaled)
print(f"Fuzzy-ACE ARI: {adjusted_rand_score(true_labels, labels_fcm):.4f}")
print(f"Fuzzy-ACE NMI: {normalized_mutual_info_score(true_labels, labels_fcm):.4f}")


--- Running Proposed Fuzzy-ACE (FCM) ---
Fuzzy-ACE ARI: 0.6101
Fuzzy-ACE NMI: 0.6526
